# LAT 1 transition: v1.0


In [2]:
from deep_cartograph.deep_carto import deep_cartograph 
import importlib.resources as resources
from deep_cartograph import data

from IPython.display import display, HTML
import matplotlib.pyplot as plt
from decimal import Decimal
import pandas as pd
import numpy as np
import shutil
import logging
import yaml
import time
import os

# Get the path to the data
data_folder = resources.files(data)

# Set logging level
logging.basicConfig(level=logging.INFO)

def create_time_evolution(proj_data_path: str, output_path: str):
    """
    Create a time evolution plot for the CV with customized font sizes.
    """

    # Read projected data
    proj_data = pd.read_csv(proj_data_path)

    # Create time array
    time_array = np.arange(len(proj_data))
    
    # Find number of components
    n_components = proj_data.shape[1]

    # --- Custom Font Sizes ---
    LABEL_SIZE = 18
    TITLE_SIZE = 18
    TICK_SIZE = 18
    LEGEND_SIZE = 18

    # Create plot
    plt.figure(figsize=(10, 8))
    for i in range(n_components):
        plt.plot(time_array, proj_data.iloc[:, i], label=f'CV {i+1}')
    
    # Apply Font Sizes
    plt.xlabel('Time step', fontsize=LABEL_SIZE)
    plt.ylabel('Projected CV', fontsize=LABEL_SIZE)
    plt.title('Time Evolution of Projected CV', fontsize=TITLE_SIZE)
    
    # Apply tick label sizes
    plt.xticks(fontsize=TICK_SIZE)
    plt.yticks(fontsize=TICK_SIZE)
    
    # Apply legend size
    plt.legend(fontsize=LEGEND_SIZE)
    
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

def show_results(output_folder: str, model_name: str, system: str):
    """
    Show the results for a specific model trained with deep cartograph

    Inputs
    ------

        output_folder   (str):          path to the output folder
        model_name      (str):          name of the model
    """

    def show_score(score_path):
        """
        Print score path in a nice format 

        Input
        -----

            score_path: path to the score file
        """

        # Read score
        with open(score_path, 'r') as file:
            score = file.read()

        # Print score in scientific notation
        print(f"Final model score: {Decimal(score):.4E}")

    def show_eigenvalues(eig_path):
        """
        Print eigenvalues in a nice format

        Input
        -----

            eig_path: path to the eigenvalues file
        """

        # Read eigenvalues
        with open(eig_path, 'r') as file:
            eigenvalues = file.readlines()

        # Print eigenvalues
        for i, eig in enumerate(eigenvalues):
            print(f"Eigenvalue {i+1}: {Decimal(eig):.4E}")
    
    
    # Show output folder
    print(f"Output folder: {output_folder}")

    # Training folder
    train_cv_folder = os.path.join(output_folder, 'train_colvars')

    # Check that training folder exists
    if not os.path.exists(train_cv_folder):
        print("Training folder not found")
        return

    traj_projection_folder = os.path.join(output_folder, 'traj_projection')
    
    # Check that traj projection folder exists
    if not os.path.exists(traj_projection_folder):
        print("Trajectory projection folder not found")
        return
    
    # Model folder
    model_folder = os.path.join(train_cv_folder, model_name)

    # Check that model folder exists
    if not os.path.exists(model_folder):
        print("Model folder not found")
        return

    training_folder = os.path.join(model_folder, 'training')

    # Check that training folder exists
    if not os.path.exists(training_folder):
        print("Training folder not found")
        return

    traj_data_folder = os.path.join(model_folder, 'traj_data', system)

    # Check that traj data folder exists
    if not os.path.exists(traj_data_folder):
        print("Trajectory data folder not found")
        return

    # Show score if any
    if model_name in ['vae', 'ae', 'deep_tica']:
        score_path = os.path.join(training_folder, 'model_score.txt')
        if os.path.exists(score_path):
            show_score(score_path)
        else:
            print("Warning: Score file not found")

    # Show eigenvalues if any
    if model_name == 'deep_tica':
        eig_path = os.path.join(training_folder, 'eigenvalues.txt')
        if os.path.exists(eig_path):
            show_eigenvalues(eig_path)
        else:
            print("Warning: Eigenvalues file not found")

    # Path to projected training data
    proj_training_data_path = os.path.join(traj_data_folder, 'projected_trajectory.csv')
    
    proj_train_plot = os.path.join(traj_data_folder, 'projected_trajectory.png')
    if os.path.exists(proj_training_data_path):
        # Create a time evolution plot for the CV
        create_time_evolution(proj_training_data_path, proj_train_plot)
    
    # Paths to images
    trajectory = os.path.join(traj_data_folder, 'trajectory.png')
    loss = os.path.join(training_folder, 'loss.png')
    beta_loss = os.path.join(training_folder, 'vae_kl_reconstruction_loss.png')
    eigenvalues = os.path.join(training_folder, 'eigenvalues.png')
    fes = os.path.join(traj_projection_folder, model_name, 'fes', f'fes_{model_name}_1', 'fes.png')
    
    paths = [trajectory, loss, beta_loss, eigenvalues, proj_train_plot, fes]

    # Generate HTML image tags
    timestamp = int(time.time()) # Add timestamp to avoid caching
    images_html = [f'<img src="{path}?{timestamp}" style="width: 600px; margin-right: 10px;">' for path in paths if os.path.exists(path)]

    if not images_html:
        print("Warning: No images to display.")
        return

    # Display images
    display(HTML(''.join(images_html)))

/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [ ]:
# Input trajectory and topology
version = "GOdMD_v1"
training_replica = "_1"
system_name = "GOdMD_6IRS_7DSQ"
GOdMD_input_path = f"{data_folder}/lat1/input/GOdMD_v1"
traj_data = os.path.join(GOdMD_input_path, 'trajs') 
top_data = os.path.join(GOdMD_input_path, 'tops')

MD_input_path = f"{data_folder}/lat1/input/MD_equil"
sup_traj_data = os.path.join(MD_input_path, 'trajs')
sup_top_data = os.path.join(MD_input_path, 'tops')
feature_list = ['torsions', 'coordinates', 'distances']

cv_models = ['vae', 'deep_tica']

for features in feature_list:
    
    config_path = f"{data_folder}/lat1/input/{features}_config.yml"
    with open(config_path) as config_file:
        configuration = yaml.load(config_file, Loader = yaml.FullLoader)

    # Output folder for the full workflow
    output_folder = f"{data_folder}/lat1/output/tests/{version}/{features}{training_replica}"

    # Clean output folder
    #if os.path.exists(output_folder):
    #    shutil.rmtree(output_folder)

    # Run workflow
    deep_cartograph(
        configuration = configuration,
        seed_trajectory_data = traj_data,
        seed_topology_data = top_data,
        supplementary_traj_data = sup_traj_data,
        supplementary_top_data = sup_top_data,
        waypoints_data = sup_top_data,
        cvs = cv_models,
        dimension = 1,
        restart = True,
        output_folder = output_folder)


INFO:deep_cartograph:================
INFO:deep_cartograph:Analyze geometry
INFO:deep_cartograph:================
INFO:deep_cartograph:Elapsed time (Analyze geometry): 00 h 00 min 00 s
INFO:deep_cartograph:=======================
INFO:deep_cartograph:Trajectory Augmentation
INFO:deep_cartograph:=======================
INFO:deep_cartograph:Processing trajectory: GOdMD_6IRS_7DSQ
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This b

KLAAnnealing initialized with type=sigmoid, start_beta=1e-06, max_beta=0.01)


INFO: `Trainer.fit` stopped: `max_epochs=1000` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1000` reached.
INFO:deep_cartograph.modules.cv_learning.cv_calculator:Successfully loaded the 'last' model from: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/torsions_1/train_colvars/vae/training/checkpoints/last.ckpt
INFO:deep_cartograph.modules.cv_learning.cv_calculator:  -> Best Overall Score:      0.02143
INFO:deep_cartograph.modules.cv_learning.cv_calculator:  -> Best Post-Annealing Score: 0.05001
INFO:deep_cartograph.modules.cv_learning.cv_calculator:  -> Last Model Score:          0.05112
INFO:deep_cartograph.modules.common.common:Starting compression to '/home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/torsions_1/train_colvars/vae/training/training_metrics.zip'...
INFO:deep_cartograph.modules.common.common:Successfully created zip file: '/home/pnavarro/repos/N

KEY:  data


Plotting only the first 20 features out of 115.
Plotting only the first 20 features out of 115.


INFO:deep_cartograph.tools.train_colvars.train_colvars_workflow:Processing trajectory: GOdMD_6IRS_7DSQ
INFO:deep_cartograph.tools.train_colvars.train_colvars_workflow:Corresponding colvars file: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/torsions_1/compute_features/GOdMD_6IRS_7DSQ_augmented_pchip/colvars.dat
INFO:deep_cartograph.tools.train_colvars.train_colvars_workflow:Corresponding topology file: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/torsions_1/traj_augmentation/GOdMD_6IRS_7DSQ_augmented_pchip.pdb
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/pnavarro/.c

...

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

In [ ]:
from deep_cartograph.modules.common import read_list

training_replica = "_1"
system_name = "GOdMD_6IRS_7DSQ"

for features in feature_list:
    print("################################################################################################")
    print(f"Features: {features.upper()}")
    output_folder = f"{data_folder}/lat1/output/{version}/tests/{features}{training_replica}"
    path_full_set_features = os.path.join(output_folder, 'filter_features', 'all_features.txt')
    path_filtered_set_features = os.path.join(output_folder, 'filter_features', 'filtered_features.txt')
    full_set_features = read_list(path_full_set_features)
    filtered_set_features = read_list(path_filtered_set_features)
    print(f"Number of features in the full set: {len(full_set_features)}")
    print(f"Number of features in the filtered set: {len(filtered_set_features)}")
    for model in cv_models:
        print(f"Results for features: {features.upper()}")
        print(f"Results for model: {model.upper()}")
        show_results(output_folder, model_name = model, system = system_name)
    print("################################################################################################")

################################################################################################
Features: TORSIONS
Number of features in the full set: 908
Number of features in the filtered set: 65
Results for features: TORSIONS
Results for model: VAE
Output folder: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/torsions
Final model score: 3.6024E-2


Results for features: TORSIONS
Results for model: DEEP_TICA
Output folder: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/torsions
Final model score: -1.0211E+0
Eigenvalue 1: 1.0105E+0


################################################################################################
################################################################################################
Features: COORDINATES
Number of features in the full set: 1371
Number of features in the filtered set: 879
Results for features: COORDINATES
Results for model: VAE
Output folder: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/coordinates
Final model score: 4.8077E-1


Results for features: COORDINATES
Results for model: DEEP_TICA
Output folder: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/coordinates
Final model score: -1.0347E+0
Eigenvalue 1: 1.0172E+0


################################################################################################
################################################################################################
Features: DISTANCES
Number of features in the full set: 2356
Number of features in the filtered set: 188
Results for features: DISTANCES
Results for model: VAE
Output folder: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/distances
Final model score: 1.3576E-2


Results for features: DISTANCES
Results for model: DEEP_TICA
Output folder: /home/pnavarro/repos/NBDsoftware/deep_cartograph/deep_cartograph/data/lat1/output/GOdMD_v1/distances
Final model score: -1.0191E+0
Eigenvalue 1: 1.0095E+0


################################################################################################


In [ ]:
# Augment the trajectory and add noise for an interpolation test 

# Should this be the validation data? 